## The `sshd` server and file transfer

**The server side.** The **`sshd`** daemon listens on port 22; its config is **`/etc/ssh/sshd_config`**. The settings you'll actually touch:

```
PermitRootLogin prohibit-password    # root only by key, never password (or 'no' to disable)
PasswordAuthentication no            # keys only — reject password attempts
AllowUsers alice bob                 # whitelist who may log in
MaxAuthTries 3
```

⚠️ **Always validate and reload before disconnecting.** A broken config won't drop your current session but *will* refuse new ones:

```bash
sudo sshd -t                 # syntax check — silent if OK
sudo systemctl reload sshd   # apply; existing sessions stay alive
```

The safe drill: reload, then open a **second** SSH session to confirm it works *before* closing the first. Production hardening: disable password auth, restrict root, firewall port 22, add `fail2ban`, watch `/var/log/auth.log`.

**File transfer** rides over SSH, inheriting its auth and encryption. **`scp`** is `cp` across machines (note capital **`-P`** for port, unlike ssh's `-p`):

```bash
scp file.txt user@host:/path/       # local → remote
scp -r dir/ user@host:/path/        # recursive
```

For anything real, use **`rsync`** — incremental (only changed bytes), resumable, ideal for backups:

```bash
rsync -avz --delete ~/work/ user@host:/backups/work/
```

**`-a`** archive mode (recursive + preserve), **`-z`** compress, **`--delete`** true mirror. ⚠️ **The trailing-slash gotcha:** `src/` copies the *contents* of `src`; `src` (no slash) copies the *directory itself* into the destination. Many broken backups start here.
